In [ ]:
# ============================================================
# 1. Imports
# ============================================================

import os
import time
import joblib
import numpy as np
import pandas as pd

from xgboost import XGBRegressor

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV
)

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)


# ============================================================
# 2. Load dataset
# ============================================================

file_name = "XGBoost mockup data.csv"

df = pd.read_csv(
    file_name,
    sep=",",
    encoding="cp1255",
    engine="python"
)


df.columns = df.columns.str.strip()

print("Original dataset shape:", df.shape)
print("Loaded file:", file_name)


# ============================================================
# 3. Convert fetal_sex
# ============================================================


df["fetal_sex"] = (
    df["fetal_sex"]
    .astype(str)
    .str.strip()
    .replace({
        "זכר": 1,
        "נקבה": 0,
        "1": 1,
        "0": 0
    })
)

df["fetal_sex"] = pd.to_numeric(
    df["fetal_sex"],
    errors="coerce"
)

print("\nfetal_sex values:")
print(df["fetal_sex"].value_counts(dropna=False))


# ============================================================
# 4. Convert dm_status to dummy variables
# ============================================================

df["dm_status"] = pd.to_numeric(
    df["dm_status"],
    errors="coerce"
).astype("Int64")

df = pd.get_dummies(
    df,
    columns=["dm_status"],
    prefix="dm",
    dtype=int
)


for column in ["dm_0", "dm_1", "dm_2"]:
    if column not in df.columns:
        df[column] = 0

print("\nDM columns:")
print([
    column
    for column in df.columns
    if column.startswith("dm_")
])


# ============================================================
# 5. Feature engineering
# ============================================================

df["maternal_weight_gain"] = (
    pd.to_numeric(
        df["current_weight"],
        errors="coerce"
    )
    -
    pd.to_numeric(
        df["pre_pregnancy_weight"],
        errors="coerce"
    )
)


df["maternal_weight_gain_is_missing"] = (
    (
        pd.to_numeric(
            df["curr_weight_is_missing"],
            errors="coerce"
        ) == 1
    )
    |
    (
        pd.to_numeric(
            df["pre_weight_is_missing"],
            errors="coerce"
        ) == 1
    )
).astype(int)


# ============================================================
# 6. Define features and target
# ============================================================

features = [
    "gestational_age",
    "smoking",
    "alcohol",
    "drugs",
    "pregnancy_age",

    "ab",
    "cs",
    "eup",
    "g",
    "lc",
    "p",
    "vbac",

    "dm_0",
    "dm_1",
    "dm_2",

    "mother_height",
    "height_is_missing",

    "pre_pregnancy_weight",
    "pre_weight_is_missing",

    "current_weight",
    "curr_weight_is_missing",

    "maternal_weight_gain",
    "maternal_weight_gain_is_missing",

    "bmi",
    "bmi_is_missing",

    "us_estimated_weight",
    "clinical_estimated_weight",

    "fetal_sex",

    "avg_previous_birth_weights",
    "avg_previous_is_missing"
]

target = "birth_weight"


# ============================================================
# 7. Check that all required columns exist
# ============================================================

missing_columns = [
    column
    for column in features + [target]
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        "The following columns are missing from the CSV: "
        f"{missing_columns}"
    )

print("\nAll required columns exist.")


# ============================================================
# 8. Convert features and target to numeric
# ============================================================

X = df[features].copy()

for column in features:
    X[column] = pd.to_numeric(
        X[column],
        errors="coerce"
    )

y = pd.to_numeric(
    df[target],
    errors="coerce"
)


# ============================================================
# 9. Remove rows with remaining missing values
# ============================================================

model_data = pd.concat(
    [
        X,
        y.rename(target)
    ],
    axis=1
)

print("\nMissing values before final filtering:")
print(
    model_data
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(30)
)

rows_before = len(model_data)

model_data = model_data.dropna().copy()

rows_after = len(model_data)

print("\nRows before cleaning:", rows_before)
print("Rows after cleaning:", rows_after)
print("Rows removed:", rows_before - rows_after)


# ============================================================
# 10. Final X and y
# ============================================================

X = model_data[features]
y = model_data[target]

print("\nFinal X shape:", X.shape)
print("Final y shape:", y.shape)

print("\nNon-numeric columns:")
print(
    X.select_dtypes(
        exclude=np.number
    ).columns.tolist()
)

print("\nTotal missing values in X:")
print(X.isna().sum().sum())


# ============================================================
# 11. Train/Test split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)


# ============================================================
# 12. Initialize XGBoost
# ============================================================

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="mae",
    random_state=42,
    n_jobs=1,
    tree_method="hist"
)


# ============================================================
# 13. Hyperparameter ranges
# ============================================================

param_grid = {
    "n_estimators": [
        300,
        500,
        700,
        1000
    ],

    "max_depth": [
        2,
        3,
        4,
        5,
        6
    ],

    "learning_rate": [
        0.01,
        0.02,
        0.03,
        0.05,
        0.08
    ],

    "subsample": [
        0.7,
        0.8,
        0.9,
        1.0
    ],

    "colsample_bytree": [
        0.6,
        0.7,
        0.8,
        0.9,
        1.0
    ],

    "min_child_weight": [
        1,
        3,
        5,
        7,
        10
    ],

    "gamma": [
        0,
        0.05,
        0.1,
        0.2,
        0.5
    ],

    "reg_alpha": [
        0,
        0.001,
        0.01,
        0.1,
        1
    ],

    "reg_lambda": [
        1,
        2,
        5,
        10,
        20
    ]
}


# ============================================================
# 14. RandomizedSearchCV
# ============================================================

search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=50,
    cv=5,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

search_start = time.perf_counter()

search.fit(
    X_train,
    y_train
)

search_end = time.perf_counter()

search_time = search_end - search_start


# ============================================================
# 15. Best parameters
# ============================================================

print("\n================================")
print("Best XGBoost Parameters")
print("================================")

for parameter, value in search.best_params_.items():
    print(f"{parameter}: {value}")

print(
    f"\nBest Cross-Validation MAE: "
    f"{-search.best_score_:.2f} grams"
)

print(
    f"Search time: "
    f"{search_time / 60:.2f} minutes"
)


# ============================================================
# 16. Best model and predictions
# ============================================================

model = search.best_estimator_

prediction_start = time.perf_counter()

y_pred = model.predict(
    X_test
)

prediction_end = time.perf_counter()

prediction_time = (
    prediction_end
    - prediction_start
)


# ============================================================
# 17. Final metrics
# ============================================================

test_mae = mean_absolute_error(
    y_test,
    y_pred
)

test_mse = mean_squared_error(
    y_test,
    y_pred
)

test_rmse = np.sqrt(
    test_mse
)

test_r2 = r2_score(
    y_test,
    y_pred
)

y_test_array = y_test.to_numpy()

non_zero_mask = (
    y_test_array != 0
)

test_mape = np.mean(
    np.abs(
        (
            y_test_array[non_zero_mask]
            - y_pred[non_zero_mask]
        )
        / y_test_array[non_zero_mask]
    )
) * 100


print("\n================================")
print("XGBoost Final Test Results")
print("================================")

print(f"MAE:  {test_mae:.2f} grams")
print(f"RMSE: {test_rmse:.2f} grams")
print(f"R²:   {test_r2:.4f}")
print(f"MAPE: {test_mape:.2f}%")
print(f"Prediction time: {prediction_time:.4f} seconds")


# ============================================================
# 18. Feature importance
# ============================================================

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
})

importance_sum = (
    feature_importance["importance"].sum()
)

if importance_sum > 0:
    feature_importance["importance_percent"] = (
        feature_importance["importance"]
        / importance_sum
        * 100
    )
else:
    feature_importance["importance_percent"] = 0

feature_importance = (
    feature_importance
    .sort_values(
        by="importance_percent",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n================================")
print("XGBoost Feature Importance")
print("================================")

print(
    feature_importance[
        [
            "feature",
            "importance_percent"
        ]
    ].to_string(
        index=False,
        formatters={
            "importance_percent":
                lambda value: f"{value:.2f}%"
        }
    )
)

feature_importance.to_csv(
    "xgboost_feature_importance.csv",
    index=False
)

print(
    "\nxgboost_feature_importance.csv "
    "saved successfully"
)


# ============================================================
# 19. Create predictions table
# ============================================================

results = X_test.copy()

results["actual_birth_weight"] = (
    y_test.to_numpy()
)

results["predicted_birth_weight"] = (
    y_pred
)

results["absolute_error"] = np.abs(
    results["actual_birth_weight"]
    - results["predicted_birth_weight"]
)

results["signed_error"] = (
    results["predicted_birth_weight"]
    - results["actual_birth_weight"]
)

results["percentage_error"] = (
    results["absolute_error"]
    / results["actual_birth_weight"]
    * 100
)


# ============================================================
# 20. Save all predictions
# ============================================================

results.to_csv(
    "xgboost_predictions.csv",
    index=False
)

print(
    "xgboost_predictions.csv "
    "saved successfully"
)


# ============================================================
# 21. Worst 50 predictions
# ============================================================

worst_predictions = (
    results
    .sort_values(
        by="absolute_error",
        ascending=False
    )
    .head(50)
)

display(worst_predictions)

worst_predictions.to_csv(
    "xgboost_worst_predictions.csv",
    index=False
)

print(
    "xgboost_worst_predictions.csv "
    "saved successfully"
)


# ============================================================
# 22. Compare missing indicators
# ============================================================

missing_indicator_columns = [
    "height_is_missing",
    "pre_weight_is_missing",
    "curr_weight_is_missing",
    "bmi_is_missing",
    "avg_previous_is_missing",
    "maternal_weight_gain_is_missing"
]

existing_missing_columns = [
    column
    for column in missing_indicator_columns
    if column in X_test.columns
]

print("\nMissing indicators in Worst 50:")

print(
    worst_predictions[
        existing_missing_columns
    ].mean() * 100
)

print("\nMissing indicators in all test rows:")

print(
    X_test[
        existing_missing_columns
    ].mean() * 100
)


# ============================================================
# 23. Save metrics
# ============================================================

metrics = pd.DataFrame({
    "model": ["XGBoost"],

    "cross_validation_mae": [
        -search.best_score_
    ],

    "test_mae_grams": [
        test_mae
    ],

    "test_rmse_grams": [
        test_rmse
    ],

    "test_r2": [
        test_r2
    ],

    "test_mape_percent": [
        test_mape
    ],

    "search_time_seconds": [
        search_time
    ],

    "prediction_time_seconds": [
        prediction_time
    ],

    "training_rows": [
        len(X_train)
    ],

    "test_rows": [
        len(X_test)
    ],

    "number_of_features": [
        X_train.shape[1]
    ]
})

metrics.to_csv(
    "xgboost_metrics.csv",
    index=False
)

print(
    "xgboost_metrics.csv "
    "saved successfully"
)


# ============================================================
# 24. Save model
# ============================================================

joblib.dump(
    {
        "model": model,
        "features": features,
        "best_parameters": search.best_params_,
        "cross_validation_mae": -search.best_score_,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "test_r2": test_r2,
        "test_mape": test_mape
    },
    "birth_weight_xgboost_model.joblib"
)

print(
    "birth_weight_xgboost_model.joblib "
    "saved successfully"
)


# ============================================================
# 25. Output folder
# ============================================================

print("\nFiles were saved in:")
print(os.getcwd())